In [1]:
import sqlite3
import pandas as pd

# Connect to Chinook database
conn = sqlite3.connect('chinook.db')

print("Connected to Chinook database")

Connected to Chinook database


In [2]:
# Configure pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

In [3]:
SCHEMA REFERENCE
Key Tables andColumns
tracks

TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer
Milliseconds, Bytes, UnitPrice

albums

AlbumId, Title, ArtistId

artists

ArtistId, Name

customers

CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId

invoices

InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total

invoice_items

InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity

genres

GenreId, Name

media_types

MediaTypeId, Name

employees

EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email

SyntaxError: invalid syntax (1543158634.py, line 1)

In [8]:
# Example 1: Show tracks ranked by price within each genre
# Tied prices get same rank, next rank is skipped

                    # RANK()

query = """
        SELECT
            Genre.Name AS Genre,
            Track.Name AS Track,
            Track.UnitPrice,
            RANK() OVER (PARTITION BY Genre.GenreId ORDER BY Track.UnitPrice DESC) AS PriceRank
        FROM Track
        JOIN Genre ON Track.GenreId = Genre.GenreId;
"""

result = pd.read_sql_query(query, conn)
print(result)

          Genre                          Track  UnitPrice  PriceRank
0          Rock  For Those About To Rock (W...       0.99          1
1          Rock              Balls to the Wall       0.99          1
2          Rock                Fast As a Shark       0.99          1
3          Rock              Restless and Wild       0.99          1
4          Rock           Princess of the Dawn       0.99          1
...         ...                            ...        ...        ...
3498  Classical  Pini Di Roma (Pinien Von R...       0.99          1
3499  Classical  String Quartet No. 12 in C...       0.99          1
3500  Classical  L'orfeo, Act 3, Sinfonia (...       0.99          1
3501  Classical  Quintet for Horn, Violin, ...       0.99          1
3502      Opera  Die Zauberflöte, K.620: "D...       0.99          1

[3503 rows x 4 columns]


In [9]:
# Example 2: Same as above but with DENSE_RANK - no gaps in ranking


                            # DENSE_RANK()

query = """
        SELECT
            Genre.Name AS Genre,
            Track.Name AS Track,
            Track.UnitPrice,
            DENSE_RANK() OVER (PARTITION BY Genre.GenreId ORDER BY Track.UnitPrice DESC) AS PriceRank
        FROM Track
        JOIN Genre ON Track.GenreId = Genre.GenreId
        
"""

result = pd.read_sql_query(query, conn)
print(result)

          Genre                          Track  UnitPrice  PriceRank
0          Rock  For Those About To Rock (W...       0.99          1
1          Rock              Balls to the Wall       0.99          1
2          Rock                Fast As a Shark       0.99          1
3          Rock              Restless and Wild       0.99          1
4          Rock           Princess of the Dawn       0.99          1
...         ...                            ...        ...        ...
3498  Classical  Pini Di Roma (Pinien Von R...       0.99          1
3499  Classical  String Quartet No. 12 in C...       0.99          1
3500  Classical  L'orfeo, Act 3, Sinfonia (...       0.99          1
3501  Classical  Quintet for Horn, Violin, ...       0.99          1
3502      Opera  Die Zauberflöte, K.620: "D...       0.99          1

[3503 rows x 4 columns]


In [11]:
# Example 3: Assign unique sequential number to each customer's invoice
# Even if invoice amounts are identical, each gets unique number


                                # ROW_NUMBER()



query = """
            SELECT
                Customer.FirstName, 
                Invoice.InvoiceDate,
                Invoice.Total,
                ROW_NUMBER() OVER (PARTITION BY Customer.CustomerId ORDER BY Invoice.InvoiceDate) AS InvoiceNumber
            FROM Customer
            JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
            WHERE Customer.FirstName = 'Frank'
            ORDER BY Invoice.InvoiceDate;
        
"""

result = pd.read_sql_query(query, conn)
print(result)

# Notice: Every row gets unique number (1, 2, 3, 4....)
# -- Use this when you need "exactly the Nth row" per group

   FirstName          InvoiceDate  Total  InvoiceNumber
0      Frank  2021-02-19 00:00:00   0.99              1
1      Frank  2022-02-08 00:00:00   1.98              1
2      Frank  2022-03-21 00:00:00  15.86              2
3      Frank  2022-08-13 00:00:00   1.98              2
4      Frank  2022-09-23 00:00:00  13.86              3
5      Frank  2022-11-19 00:00:00   8.91              3
6      Frank  2023-05-24 00:00:00   8.91              4
7      Frank  2024-06-25 00:00:00   1.98              4
8      Frank  2024-09-27 00:00:00   7.96              5
9      Frank  2024-12-28 00:00:00   1.98              5
10     Frank  2024-12-30 00:00:00   5.94              6
11     Frank  2025-04-01 00:00:00   3.96              6
12     Frank  2025-07-04 00:00:00   5.94              7
13     Frank  2025-08-20 00:00:00   0.99              7


In [13]:
# Example 4: Compare each invoice to the previous invoice for same customer

                                # lAG()


query = """
            SELECT 
                Customer.FirstName,
                Invoice.InvoiceDate,
                Invoice.Total AS CurrentInvoice,
                LAG(Invoice.Total, 1) OVER (PARTITION BY Customer.CustomerId ORDER BY Invoice.InvoiceDate) AS PreviousInvoice,
                Invoice.Total - LAG(Invoice.Total, 1) OVER (PARTITION BY Customer.CustomerId ORDER BY Invoice.InvoiceDate) AS Difference
            FROM Customer
            JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
            WHERE Customer.FirstName = 'Frank'
            ORDER BY Invoice.InvoiceDate;
           
        
"""

result = pd.read_sql_query(query, conn)
print(result)

   FirstName          InvoiceDate  CurrentInvoice  PreviousInvoice  Difference
0      Frank  2021-02-19 00:00:00            0.99              NaN         NaN
1      Frank  2022-02-08 00:00:00            1.98              NaN         NaN
2      Frank  2022-03-21 00:00:00           15.86             1.98       13.88
3      Frank  2022-08-13 00:00:00            1.98             0.99        0.99
4      Frank  2022-09-23 00:00:00           13.86             1.98       11.88
5      Frank  2022-11-19 00:00:00            8.91            15.86       -6.95
6      Frank  2023-05-24 00:00:00            8.91            13.86       -4.95
7      Frank  2024-06-25 00:00:00            1.98             8.91       -6.93
8      Frank  2024-09-27 00:00:00            7.96             1.98        5.98
9      Frank  2024-12-28 00:00:00            1.98             8.91       -6.93
10     Frank  2024-12-30 00:00:00            5.94             7.96       -2.02
11     Frank  2025-04-01 00:00:00            3.96   

In [14]:
# Example 5: Compare each invoice to the NEXT invoice for the same customer

                                # LEAD()


query = """
            SELECT 
                Customer.FirstName,
                Invoice.InvoiceDate,
                Invoice.Total AS CurrentInvoice,
                LEAD(Invoice.Total, 1) OVER (PARTITION BY Customer.CustomerId ORDER BY Invoice.InvoiceDate) AS NextInvoice,
                LEAD(Invoice.Total, 1) OVER (PARTITION BY Customer.CustomerId ORDER BY Invoice.InvoiceDate) - Invoice.Total AS ChangeToNext
            FROM Customer
            JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
            WHERE Customer.FirstName = 'Frank'
            ORDER BY Invoice.InvoiceDate;
                
            
           
        
"""

result = pd.read_sql_query(query, conn)
print(result)

# - LEAD(Column, 1) gets value from 1 row ahead (next row)
# - Last row has no next, so LEAD returns NULL
# - Opposite of LAG - looks forward instead of backward

   FirstName          InvoiceDate  CurrentInvoice  NextInvoice  ChangeToNext
0      Frank  2021-02-19 00:00:00            0.99         1.98          0.99
1      Frank  2022-02-08 00:00:00            1.98        15.86         13.88
2      Frank  2022-03-21 00:00:00           15.86         8.91         -6.95
3      Frank  2022-08-13 00:00:00            1.98        13.86         11.88
4      Frank  2022-09-23 00:00:00           13.86         8.91         -4.95
5      Frank  2022-11-19 00:00:00            8.91         1.98         -6.93
6      Frank  2023-05-24 00:00:00            8.91         1.98         -6.93
7      Frank  2024-06-25 00:00:00            1.98         7.96          5.98
8      Frank  2024-09-27 00:00:00            7.96         5.94         -2.02
9      Frank  2024-12-28 00:00:00            1.98         3.96          1.98
10     Frank  2024-12-30 00:00:00            5.94         0.99         -4.95
11     Frank  2025-04-01 00:00:00            3.96         5.94          1.98

In [17]:
# Example 6: Divide all tracks into 4 price quartiles (buckets)

                                # NTILE()


query = """
            SELECT
                Track.Name,
                Track.UnitPrice,
                NTILE(4) OVER (ORDER BY Track.UnitPrice) AS PriceQuartile
            FROM Track;
                
            
           
        
"""

result = pd.read_sql_query(query, conn)
print(result)

# - NTILE(4) divides data into 4 equal groups
# - Quartile 1 = cheapest 25%, Quartile 4 = most expensive 25%
# - Use binning data into percentile groups

                               Name  UnitPrice  PriceQuartile
0     For Those About To Rock (W...       0.99              1
1                 Balls to the Wall       0.99              1
2                   Fast As a Shark       0.99              1
3                 Restless and Wild       0.99              1
4              Princess of the Dawn       0.99              1
...                             ...        ...            ...
3498  There's No Place Like Home...       1.99              4
3499  There's No Place Like Home...       1.99              4
3500  There's No Place Like Home...       1.99              4
3501                 Branch Closing       1.99              4
3502                     The Return       1.99              4

[3503 rows x 3 columns]


In [18]:
# Example 7: Show each track's percentile rank by length within its genre


                                # PERCENT_RANK()


query = """
            SELECT 
                Genre.Name AS Genre,
                Track.Name AS Track,
                Track.Milliseconds / 60000.0 AS MINUTES,
                ROUND(PERCENT_RANK() OVER (PARTITION BY Genre.GenreId ORDER BY Track.Milliseconds), 2) AS PercentileRank
            FROM Track
            JOIN Genre ON Track.GenreId = Genre.GenreId
            WHERE Genre.Name = 'Jazz'
            ORDER BY Track.Milliseconds DESC
            LIMIT 10;
                
            
           
        
"""

result = pd.read_sql_query(query, conn)
print(result)

# PERCENT_RANK returns 0.0 to 1.0 ( 0% to 100%)
# 0.95 means "longer than 95% of tracks in this genre"
# Use to find "top 10%" by filtering WHERE PERCENT_RANK >= 0.9

  Genre                          Track    MINUTES  PercentileRank
0  Jazz      My Funny Valentine (Live)  15.125333            1.00
1  Jazz     Miles Runs The Voodoo Down  14.066067            0.99
2  Jazz                        Walkin'  13.456533            0.98
3  Jazz                       Outbreak  10.987100            0.98
4  Jazz                        Stratus   9.701433            0.97
5  Jazz                        So What   9.400150            0.96
6  Jazz    Someday My Prince Will Come   9.067967            0.95
7  Jazz                She Wears Black   8.811100            0.95
8  Jazz  Petits Machins (Little Stuff)   8.123200            0.94
9  Jazz              Bye Bye Blackbird   7.933383            0.93


In [21]:
# Example 8: Calculate 7-day moving average of daily revenue


                                # Moving Average (Window Frame)
query = """
            SELECT
                InvoiceDate,
                SUM(Total) AS DailyRevenue,
                AVG(SUM(Total)) OVER(
                    ORDER BY InvoiceDate
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
                ) AS MovingAVG7Day
            FROM Invoice
            GROUP BY InvoiceDate
            ORDER BY InvoiceDate;
                
            
           
        
"""

result = pd.read_sql_query(query, conn)
print(result)


# ROWS BETWEEN 6 PRECEDING AND CURRENT ROW = last 7 rows (including current)
# Smooths out daily fluctuations to show trends
# First 6 rows have fewer than 7 days, so average is based on available data

             InvoiceDate  DailyRevenue  MovingAVG7Day
0    2021-01-01 00:00:00          1.98       1.980000
1    2021-01-02 00:00:00          3.96       2.970000
2    2021-01-03 00:00:00          5.94       3.960000
3    2021-01-06 00:00:00          8.91       5.197500
4    2021-01-11 00:00:00         13.86       6.930000
..                   ...           ...            ...
349  2025-12-05 00:00:00          3.96       7.654286
350  2025-12-06 00:00:00          5.94       7.937143
351  2025-12-09 00:00:00          8.91       8.361429
352  2025-12-14 00:00:00         13.86       9.068571
353  2025-12-22 00:00:00          1.99       5.658571

[354 rows x 3 columns]
